# PE 估值演示：以 ORCL（Oracle）为例

本 notebook 展示从 yfinance 拉取数据，到用 PE 法计算目标价的完整过程。

目的：搞清楚 yfinance 给了我们什么 EPS，以及 PE 估值结果是否合理。

In [1]:
# ── Step 0：修复 SSL（路径含中文时需要）──────────────────────────────────────
import os, shutil, tempfile
import certifi

cert_path = certifi.where()
print(f"certifi 路径: {cert_path}")

if any(ord(c) > 127 for c in cert_path):
    tmp = os.path.join(tempfile.gettempdir(), 'cacert_valmod.pem')
    if not os.path.exists(tmp):
        shutil.copy2(cert_path, tmp)
    os.environ['CURL_CA_BUNDLE'] = tmp
    os.environ['REQUESTS_CA_BUNDLE'] = tmp
    certifi.where = lambda: tmp
    print(f"已 patch → {tmp}")
else:
    print("路径无中文，无需 patch")

certifi 路径: c:\Programming\##github related\ChengzhiZixv\公司估值相关\估值\module3\myenv2\lib\site-packages\certifi\cacert.pem
已 patch → C:\Users\31092\AppData\Local\Temp\cacert_valmod.pem


## Step 1：拉取 ORCL 原始数据

In [2]:
import yfinance as yf

t = yf.Ticker('ORCL')
info = t.info

current_price    = info.get('currentPrice') or info.get('regularMarketPrice')
trailing_pe_info = info.get('trailingPE')        # yfinance info 里的 PE（未稀释，不可靠）
trailing_eps     = info.get('trailingEps')        # yfinance info 里的 trailing EPS
forward_eps      = info.get('forwardEps')         # 分析师预期的下一年 EPS
forward_pe       = info.get('forwardPE')

print(f"当前股价          : ${current_price}")
print(f"trailingPE (info) : {trailing_pe_info:.2f}" if trailing_pe_info else "trailingPE (info) : N/A")
print(f"trailingEps (info): ${trailing_eps}"  if trailing_eps else "trailingEps (info): N/A")
print(f"forwardEps (info) : ${forward_eps}"   if forward_eps  else "forwardEps (info) : N/A")
print(f"forwardPE (info)  : {forward_pe:.2f}" if forward_pe   else "forwardPE (info)  : N/A")

当前股价          : $148.08
trailingPE (info) : 27.83
trailingEps (info): $5.32
forwardEps (info) : $7.90395
forwardPE (info)  : 18.73


## Step 2：从财报（income_stmt）提取稀释 EPS

yfinance 的 `info['trailingEps']` 有时用未稀释 EPS，与 MOOMOO 等平台不一致。  
更可靠的方式：直接从利润表读 `Diluted EPS`。

In [3]:
import pandas as pd

income = t.income_stmt

print("=== income_stmt 列（财报期）===")
print(income.columns.tolist())
print()

# 看看有哪些行（指标名）
print("=== income_stmt 所有行名 ===")
print(income.index.tolist())

=== income_stmt 列（财报期）===
[]

=== income_stmt 所有行名 ===
[]


In [4]:
# ── 年度财报 Diluted EPS ──────────────────────────────────────────────────────
diluted_eps_series = None
for key in ['Diluted EPS', 'Basic EPS']:
    if key in income.index:
        diluted_eps_series = income.loc[key]
        print(f"找到 '{key}'（年度，仅最近4期）：")
        print(diluted_eps_series)
        break

diluted_eps_latest = float(diluted_eps_series.iloc[0]) if diluted_eps_series is not None else None
if diluted_eps_latest:
    print(f"\n年度财报最新 Diluted EPS: ${diluted_eps_latest:.4f}（财年截止日见上方列名）")

# ── 季度财报 → 计算真正的 TTM EPS ────────────────────────────────────────────
print("\n" + "="*50)
print("季度财报数据（更新）")
print("="*50)

q_income = t.quarterly_income_stmt
print(f"季度财报期：{q_income.columns.tolist()}")

ttm_eps = None
for key in ['Diluted EPS', 'Basic EPS']:
    if key in q_income.index:
        q_eps = q_income.loc[key]
        print(f"\n各季度 {key}：")
        print(q_eps)
        # TTM = 最近4个季度求和
        ttm_eps = float(q_eps.iloc[:4].sum())
        print(f"\nTTM Diluted EPS（最近4季度之和）= ${ttm_eps:.4f}")
        break


季度财报数据（更新）
季度财报期：[]


## Step 3：PE 估值计算

**公式**：目标价 = PE锚定倍数 × EPS

PE锚定倍数 = 你认为这家公司合理的 PE（需要人工判断）

In [5]:
# ── 可以修改这里的参数 ────────────────────────────────────────────
pe_anchor = 25.0   # 你认为 ORCL 合理的 PE 倍数（可调）
# ─────────────────────────────────────────────────────────────────

print("=" * 55)
print("PE 估值计算（四种 EPS 来源对比）")
print("=" * 55)
print(f"PE 锚定倍数: {pe_anchor}x　　当前股价: ${current_price:.2f}")
print()

eps_sources = {
    "年度财报 Diluted EPS（ORCL财年截5月）": diluted_eps_latest,
    "季度加总 TTM EPS（最近4季之和）★推荐": ttm_eps,
    "info.trailingEps（yfinance TTM近似）": trailing_eps,
    "info.forwardEps（分析师预期下年）": forward_eps,
}

for source, eps in eps_sources.items():
    if eps and eps > 0:
        target = pe_anchor * eps
        upside = (target - current_price) / current_price
        print(f"[{source}]")
        print(f"  EPS    = ${eps:.4f}")
        print(f"  目标价 = {pe_anchor}x × ${eps:.4f} = ${target:.2f}  ({upside:+.1%})")
        print()
    else:
        print(f"[{source}] 数据不可用\n")

PE 估值计算（四种 EPS 来源对比）
PE 锚定倍数: 25.0x　　当前股价: $148.08

[年度财报 Diluted EPS（ORCL财年截5月）] 数据不可用

[季度加总 TTM EPS（最近4季之和）★推荐] 数据不可用

[info.trailingEps（yfinance TTM近似）]
  EPS    = $5.3200
  目标价 = 25.0x × $5.3200 = $133.00  (-10.2%)

[info.forwardEps（分析师预期下年）]
  EPS    = $7.9040
  目标价 = 25.0x × $7.9040 = $197.60  (+33.4%)



## Step 4：当前实际 PE 是多少？

验证：当前股价 / EPS = 实际 PE，和 yfinance 给的 trailingPE 是否一致？

In [6]:
print("=== 实际 PE 校验 ===")
print(f"yfinance info.trailingPE = {trailing_pe_info}")

if diluted_eps_latest and diluted_eps_latest > 0:
    actual_pe_from_stmt = current_price / diluted_eps_latest
    print(f"股价 / 财报Diluted EPS   = {current_price:.2f} / {diluted_eps_latest:.4f} = {actual_pe_from_stmt:.1f}x")

if trailing_eps and trailing_eps > 0:
    actual_pe_from_trailing = current_price / trailing_eps
    print(f"股价 / trailingEps       = {current_price:.2f} / {trailing_eps:.4f} = {actual_pe_from_trailing:.1f}x")

if forward_eps and forward_eps > 0:
    actual_fwd_pe = current_price / forward_eps
    print(f"股价 / forwardEps        = {current_price:.2f} / {forward_eps:.4f} = {actual_fwd_pe:.1f}x")

=== 实际 PE 校验 ===
yfinance info.trailingPE = 27.834585
股价 / trailingEps       = 148.08 / 5.3200 = 27.8x
股价 / forwardEps        = 148.08 / 7.9040 = 18.7x


## Step 5：敏感性分析 —— 不同 PE 倍数下的目标价

In [7]:
import pandas as pd

pe_range = [18, 20, 22, 25, 28, 30, 35]
eps_to_use = diluted_eps_latest or trailing_eps

if eps_to_use:
    rows = []
    for pe in pe_range:
        target = pe * eps_to_use
        upside = (target - current_price) / current_price
        rows.append({
            'PE倍数': f'{pe}x',
            'EPS': f'${eps_to_use:.2f}',
            '目标价': f'${target:.2f}',
            '当前股价': f'${current_price:.2f}',
            '涨跌幅': f'{upside:+.1%}',
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

PE倍数   EPS     目标价    当前股价    涨跌幅
 18x $5.32  $95.76 $148.08 -35.3%
 20x $5.32 $106.40 $148.08 -28.1%
 22x $5.32 $117.04 $148.08 -21.0%
 25x $5.32 $133.00 $148.08 -10.2%
 28x $5.32 $148.96 $148.08  +0.6%
 30x $5.32 $159.60 $148.08  +7.8%
 35x $5.32 $186.20 $148.08 +25.7%


---
## 结论

运行后对比：
1. 财报 Diluted EPS vs info.trailingEps —— 看是否一致
2. 你自己用的 EPS 是哪个？
3. 你认为 ORCL 合理的 PE 是多少？

找到答案后，我们就能确定系统 PE 计算的偏差来源，并修复。